import libraries

In [1]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
customers = pd.read_csv("C:\Jatin\Celebal_internship\Week-8\Ecommerce_Order_Analytics\data\customers.csv")
products = pd.read_csv("C:\Jatin\Celebal_internship\Week-8\Ecommerce_Order_Analytics\data\products.csv")
orders = pd.read_csv("C:\Jatin\Celebal_internship\Week-8\Ecommerce_Order_Analytics\data\orders.csv")
order_items = pd.read_csv("C:\Jatin\Celebal_internship\Week-8\Ecommerce_Order_Analytics\data\order_items.csv")

print("Datasets Loaded successfully")

Datasets Loaded successfully


display dataset Shapes

In [5]:
datasets = {"Customers": customers,
            "Products": products,
            "Orders": orders,
            "Order Items": order_items }

for name, df in datasets.items():
    print("-"*60)
    print(name)
    print(df.shape)

------------------------------------------------------------
Customers
(1000, 5)
------------------------------------------------------------
Products
(500, 5)
------------------------------------------------------------
Orders
(2000, 5)
------------------------------------------------------------
Order Items
(5010, 6)


preview Data

In [ ]:
customers.head()


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,O000001,P0090,3,963.41,38.85
1,2,O000001,P0189,3,742.61,43.17
2,3,O000002,P0211,2,713.58,54.15
3,4,O000002,P0065,2,841.73,35.29
4,5,O000002,P0214,4,465.98,45.23


check missing values

In [25]:
for name, df in datasets.items():

    print("\n", "="*60)

    print(name)

    print(df.isnull().sum())


Customers
customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
valid_email          0
dtype: int64

Products
product_id      0
product_name    0
category        0
subcategory     0
cost_price      0
dtype: int64

Orders
order_id       0
customer_id    0
order_date     0
status         0
region_code    0
dtype: int64

Order Items
item_id             0
order_id            0
product_id          0
quantity            0
unit_price          0
discount_percent    0
dtype: int64


check duplicate rows

In [24]:
for name, df in datasets.items():

    print(name)

    print("Duplicate Rows :", df.duplicated().sum())

    print()

Customers
Duplicate Rows : 0

Products
Duplicate Rows : 0

Orders
Duplicate Rows : 0

Order Items
Duplicate Rows : 0



dataset information

In [10]:
customers.info()
products.info()
orders.info()
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        1000 non-null   object
 1   customer_name      1000 non-null   object
 2   email              1000 non-null   object
 3   registration_date  1000 non-null   object
 4   customer_type      1000 non-null   object
dtypes: object(5)
memory usage: 39.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    object 
 1   product_name  500 non-null    object 
 2   category      500 non-null    object 
 3   subcategory   500 non-null    object 
 4   cost_price    500 non-null    float64
dtypes: float64(1), object(4)
memory usage: 19.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 t

In [11]:
customers.describe(include="all")
products.describe(include="all")
orders.describe(include="all")
order_items.describe()

,item_id,quantity,unit_price,discount_percent
count,5010.000000,5010.000000,5010.000000,5010.000000
mean,2505.500000,2.771058,504.136842,50.032509
std,1446.406755,1.783019,285.554830,28.924451
min,1.000000,-5.000000,10.060000,0.050000
25%,1253.250000,2.000000,254.787500,24.862500
50%,2505.500000,3.000000,506.595000,50.110000
75%,3757.750000,4.000000,751.655000,75.037500
max,5010.000000,5.000000,999.830000,99.970000


Clean Order Date

(YYYY-MM-DD HH:MM:SS)  -  
(DD-MM-YYYY HH:MM:SS)

Convert both into one format

In [ ]:
orders["order_date"] = pd.to_datetime( orders["order_date"], format="mixed", dayfirst=True, errors="coerce")

print("Invalid Dates :", orders["order_date"].isna().sum())

Invalid Dates : 0


Handle Missing Customer IDs

In [13]:
missing = orders["customer_id"].isna().sum()

print("Missing Customer IDs :", missing)

orders["customer_id"] = orders["customer_id"].fillna("UNKNOWN")

Missing Customer IDs : 100


Clean product names (remove extra spaces)

In [14]:
products["product_name"] = (
    products["product_name"]
    .astype(str)
    .str.strip()
    .str.title()
)

products.head()

,product_id,product_name,category,subcategory,cost_price
0,P0001,Already Kids,Clothing,Kids,209.71
1,P0002,Walk Kitchen,Home,Kitchen,297.88
2,P0003,Republican Kids,Clothing,Kids,280.15
3,P0004,No Laptop,Electronics,Laptop,337.31
4,P0005,Direction Decor,Home,Decor,78.86


Validate  email addresses

In [15]:
pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

customers["valid_email"] = customers["email"].astype(str).str.match(pattern)

invalid_emails = customers[customers["valid_email"] == False]

print("Invalid Emails :", len(invalid_emails))

invalid_emails.head()

Invalid Emails : 20


,customer_id,customer_name,email,registration_date,customer_type,valid_email
50,C00051,Elizabeth Clark,christinaturnerexample.net,2025-10-22,PREMIUM,False
62,C00063,Sarah Ashley,wwoodsexample.com,2024-09-26,REGULAR,False
63,C00064,George Shelton,tracynelsonexample.com,2024-09-11,VIP,False
145,C00146,Christopher Park,lynchdianeexample.net,2024-06-01,PREMIUM,False
184,C00185,Kevin Sherman,cooperjessicaexample.net,2023-11-27,VIP,False


Check referential  integrity

In [ ]:
invalid_orders = order_items[ ~order_items["order_id"].isin(orders["order_id"])]

print("Invalid Order References")

print(len(invalid_orders))

Invalid Order References
0


Negative Quantity

In [ ]:
negative_quantity = order_items[ order_items["quantity"] < 0 ]

print("Negative Quantity Rows")

print(len(negative_quantity))


Negative Quantity Rows
168


Discount Greater Than 100(remove invalid discount)

In [ ]:
invalid_discount = order_items[ order_items["discount_percent"] > 100 ]

print("Invalid Discount Rows")

print(len(invalid_discount))

Invalid Discount Rows
0


Future Order Dates

In [ ]:
future_orders = orders[orders["order_date"] > pd.Timestamp.today()]

print("Future Orders")

print(len(future_orders))

Future Orders
0


Remove Duplicate Rows

In [20]:
customers = customers.drop_duplicates()

products = products.drop_duplicates()

orders = orders.drop_duplicates()

order_items = order_items.drop_duplicates()

Data Quality Report

In [23]:
report = pd.DataFrame({
    "Issue":["Missing Customer ID", "Invalid Email", "Negative Quantity", "Invalid Discount",
    "Future Orders", "Invalid Order Reference"],

    "Count":[missing, len(invalid_emails), len(negative_quantity), len(invalid_discount),
    len(future_orders), len(invalid_orders)]
})

report

,Issue,Count
0,Missing Customer ID,100
1,Invalid Email,20
2,Negative Quantity,168
3,Invalid Discount,0
4,Future Orders,0
5,Invalid Order Reference,0


Save cleaned csv files 

In [22]:
os.makedirs("data/cleaned", exist_ok=True)

customers.to_csv("data/cleaned/customers_cleaned.csv", index=False)

products.to_csv("data/cleaned/products_cleaned.csv", index=False)

orders.to_csv("data/cleaned/orders_cleaned.csv", index=False)

order_items.to_csv("data/cleaned/order_items_cleaned.csv", index=False)

report.to_csv("data/cleaned/data_quality_report.csv", index=False)

print("All Cleaned Files Saved Successfully")

All Cleaned Files Saved Successfully
